<a href="https://colab.research.google.com/github/vieuxboulogne/bwaaaaah/blob/main/%D0%9A_%D0%B2%D1%81%D1%82%D1%80%D0%B5%D1%87%D0%B0%D0%BC_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
from google.colab import files
import io

!rm -rf *

uploaded = files.upload()

url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSyme8f6g0g84FPsNohz464yNYLMSoOG1icEbl_DWFHINJBD8OYOX7VQ4jQENPiN6Jo71TMduMV2fyh/pub?gid=1153986199&single=true&output=csv"
bo_file = "БО.xlsx"
weeks_file = [name for name in uploaded.keys() if name.startswith("Сводная")][0]

df = pd.read_excel(io.BytesIO(uploaded[bo_file]))
df_weeks = pd.read_excel(io.BytesIO(uploaded[weeks_file]))
reference = pd.read_csv(url)

needed = reference["БО"].dropna().tolist()
df = df[df['ФИО сотрудника'].isin(needed)]

df = df[['Ссылка на тикет','Дата закрытия тикета','Группа сотрудника','ФИО сотрудника','Группа стажа',
         'Направление категории','Тема','Рейтинг МФ','ФИО создателя формы','Дата оценки владельцем формы',
         'Тип ошибки','Ошибка','Комментарий к ошибке']].drop_duplicates(subset=['Ссылка на тикет','Дата закрытия тикета','Группа сотрудника',
         'ФИО сотрудника','Группа стажа','Рейтинг МФ','ФИО создателя формы','Дата оценки владельцем формы','Тип ошибки','Ошибка','Комментарий к ошибке'])

df['Дата оценки владельцем формы'] = pd.to_datetime(df['Дата оценки владельцем формы'])
df['Неделя'] = df['Дата оценки владельцем формы'].dt.isocalendar().week
df['Дата оценки владельцем формы'] = df['Дата оценки владельцем формы'].dt.date

stazh_map = {'1. 0-30':'0-30', '2. 31-60':'31-60','3. 61-90':'61-90','4. 91-180':'91+', '5. более 181':'91+'}
df['Группа стажа'] = df['Группа стажа'].replace(stazh_map)
df.loc[df["ФИО сотрудника"].isin(needed), "Статус"] = "Действующий ИС"

unique = df.drop_duplicates(subset=['Ссылка на тикет'])

summary = unique.groupby('ФИО сотрудника').agg(
        ФИО=('ФИО сотрудника', 'first'),
        Количество_проверок=('Ссылка на тикет', 'count'),
        Рейтинг_МФ=('Рейтинг МФ', 'mean'),
       ).sort_values(by='Количество_проверок', ascending=False)
summary = summary.reset_index(drop=True)

df['Ссылка на тикет'] = 'https://crm.o3team.ru/tickets/' + df['Ссылка на тикет'].astype(int).astype(str)

df_weeks.loc[df_weeks["ФИО сотрудника"].isin(needed), "Статус"] = "Действующий ИС"

output_filename = "Сводная_NEW.xlsx"

with pd.ExcelWriter(output_filename) as writer:
    df.to_excel(writer, sheet_name='Выгрузка',  index=False)
    summary.to_excel(writer, sheet_name='Кол-во проверок по оп', index=False)
    df_weeks.to_excel(writer, sheet_name='Выгрузка недели', index=False, startrow = 0)
    df.to_excel(writer, sheet_name='Выгрузка недели', index=False, startrow = len(df_weeks) + 1,header=False)

files.download(output_filename)

Saving БО.xlsx to БО.xlsx
Saving Сводная к встрече.xlsx to Сводная к встрече.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>